In [40]:
"""
이름: 왜용

타겟: 만 3~6세 자녀를 둔 부모

목적: 아이의 순수한 질문을 부모가 놓치지 않고,
아이 눈높이 설명과 대화로 연결할 수 있도록 돕는 부모용 호기심 학습 가이드 에이전트.

포지셔닝: 왜용은 정답을 나열하는 앱이 아니라,
부모와 아이 사이의 호기심 대화를 설계하는 코치다.

그래프(노드 순서):
[아이 질문·아이 프로필 입력] → [질문 유형·의도 분석] → [학습 주제 변환]
→ [이전 학습 조회 SQLite] → [부모용 코치 패키지: 30초/3분/대본·피할말·창의질문·다음호기심]
→ [활동 가능 여부 판단] → [안전 놀이·실험·관찰 가이드]
→ [일러스트 필요 여부 판단] → (필요 시) [이미지 생성 Tool] / (불필요 시) 생략
→ [호기심 기록 SQLite 저장]
"""

'\n이름: 왜용\n\n타겟: 만 3~6세 자녀를 둔 부모\n\n목적: 아이의 순수한 질문을 부모가 놓치지 않고,\n아이 눈높이 설명과 대화로 연결할 수 있도록 돕는 부모용 호기심 학습 가이드 에이전트.\n\n포지셔닝: 왜용은 정답을 나열하는 앱이 아니라,\n부모와 아이 사이의 호기심 대화를 설계하는 코치다.\n\n그래프(노드 순서):\n[아이 질문·아이 프로필 입력] → [질문 유형·의도 분석] → [학습 주제 변환]\n→ [이전 학습 조회 SQLite] → [부모용 코치 패키지: 30초/3분/대본·피할말·창의질문·다음호기심]\n→ [활동 가능 여부 판단] → [안전 놀이·실험·관찰 가이드]\n→ [일러스트 필요 여부 판단] → (필요 시) [이미지 생성 Tool] / (불필요 시) 생략\n→ [호기심 기록 SQLite 저장]\n'

In [ ]:
import base64
import hashlib
import sqlite3
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt
from langgraph.checkpoint.memory import MemorySaver
from typing_extensions import TypedDict, List
from langchain.chat_models import init_chat_model
from pydantic import BaseModel, Field
from openai import OpenAI
from langchain_core.tools import tool

llm = init_chat_model(model="openai:gpt-4o-mini")

conn = sqlite3.connect("learning.db", check_same_thread=False)


def _init_learning_db(connection: sqlite3.Connection) -> None:
    connection.execute(
        """
        CREATE TABLE IF NOT EXISTS learning_records (
          id INTEGER PRIMARY KEY AUTOINCREMENT,
          topic_key TEXT NOT NULL,
          learner_id TEXT DEFAULT '',
          question TEXT,
          study_title TEXT,
          core_concept TEXT,
          target_age INTEGER,
          description_excerpt TEXT,
          outcome_summary TEXT,
          created_at TEXT DEFAULT (datetime('now', 'localtime'))
        )
        """
    )
    connection.execute(
        "CREATE INDEX IF NOT EXISTS idx_learning_topic ON learning_records(topic_key)"
    )
    connection.execute(
        "CREATE INDEX IF NOT EXISTS idx_learning_learner_topic ON learning_records(learner_id, topic_key)"
    )
    connection.commit()


def _migrate_learning_db(connection: sqlite3.Connection) -> None:
    """기존 DB에 질문 카테고리 컬럼 추가."""
    cur = connection.execute("PRAGMA table_info(learning_records)")
    cols = {row[1] for row in cur.fetchall()}
    if "question_category" not in cols:
        connection.execute(
            "ALTER TABLE learning_records ADD COLUMN question_category TEXT DEFAULT ''"
        )
        connection.commit()
    connection.execute(
        "CREATE INDEX IF NOT EXISTS idx_learning_learner_category "
        "ON learning_records(learner_id, question_category)"
    )
    connection.commit()


_init_learning_db(conn)
_migrate_learning_db(conn)


def list_question_categories(learner_id: str = "") -> List[tuple]:
    """(카테고리, 건수) 목록 — 프론트 필터·사이드바용."""
    lid = (learner_id or "").strip()
    if lid:
        rows = conn.execute(
            """
            SELECT question_category, COUNT(*) FROM learning_records
            WHERE learner_id = ? AND COALESCE(question_category,'') != ''
            GROUP BY question_category ORDER BY COUNT(*) DESC
            """,
            (lid,),
        ).fetchall()
    else:
        rows = conn.execute(
            """
            SELECT question_category, COUNT(*) FROM learning_records
            WHERE COALESCE(question_category,'') != ''
            GROUP BY question_category ORDER BY COUNT(*) DESC
            """
        ).fetchall()
    return rows


def fetch_learning_records(
    learner_id: str = "",
    category: str | None = None,
    limit: int = 200,
) -> list:
    """카테고리별 질문·요약 조회. category가 None이면 전체."""
    lid = (learner_id or "").strip()
    lim = max(1, min(500, int(limit)))
    cat = (category or "").strip()

    if lid and cat:
        return conn.execute(
            """
            SELECT id, question_category, question, study_title, created_at, outcome_summary
            FROM learning_records
            WHERE learner_id = ? AND question_category = ?
            ORDER BY id DESC LIMIT ?
            """,
            (lid, cat, lim),
        ).fetchall()
    if lid:
        return conn.execute(
            """
            SELECT id, question_category, question, study_title, created_at, outcome_summary
            FROM learning_records
            WHERE learner_id = ?
            ORDER BY id DESC LIMIT ?
            """,
            (lid, lim),
        ).fetchall()
    if cat:
        return conn.execute(
            """
            SELECT id, question_category, question, study_title, created_at, outcome_summary
            FROM learning_records
            WHERE question_category = ?
            ORDER BY id DESC LIMIT ?
            """,
            (cat, lim),
        ).fetchall()
    return conn.execute(
        """
        SELECT id, question_category, question, study_title, created_at, outcome_summary
        FROM learning_records
        ORDER BY id DESC LIMIT ?
        """,
        (lim,),
    ).fetchall()


# 그래프 실행 시: config = {"configurable": {"thread_id": "demo"}}

In [42]:
class QuestionAnalysis(BaseModel):
    question_type: str = Field(
        description="질문 유형: 과학·자연 / 감정·관계 / 사회·가치 / 민감(죽음·신체·가난 등) / 일상·기타 중 짧게"
    )
    child_intent: str = Field(description="아이 질문 속 호기심·의도")
    parent_need: str = Field(description="부모에게 필요한 도움 유형")
    difficulty_level: str = Field(description="이 연령 기준 설명 난이도: 쉬움|보통|어려움")


class StudySubject(BaseModel):
    title: str = Field(description="학습 주제 제목")
    core_concept: str = Field(description="핵심 개념")
    learning_goal: str = Field(description="학습 목표")
    keywords: List[str] = Field(description="관련 키워드 목록")


class CreativeQuestionGuide(BaseModel):
    observation: str = Field(description="관찰 질문 한 문장")
    imagination: str = Field(description="상상 질문 한 문장")
    comparison: str = Field(description="비교 질문 한 문장")
    inquiry: str = Field(description="탐구 질문 한 문장")


class ParentCoachPack(BaseModel):
    short_answer_30s: str = Field(description="길에서·갑자기 물었을 때 부모가 바로 말할 30초 분량")
    story_answer_3min: str = Field(description="자기 전·여유 있을 때 조금 더 긴 이야기식 설명")
    play_and_picture_tip: str = Field(description="장난감·그림·몸으로 하는 놀이로 이어가는 짧은 팁")
    parent_read_aloud_script: str = Field(
        description="부모가 아이에게 실제로 말할 수 있게 연결한 대화 스크립트(몇 문단)"
    )
    phrases_to_avoid: str = Field(description="호기심을 꺾기 쉬운 말 2~4개 (줄바꿈으로)")
    say_instead: str = Field(description="대신 이렇게 말해보세요 — 짧은 예문들")
    follow_up_questions: List[str] = Field(description="설명 후 아이에게 되물을 질문 정확히 3개")
    creative_question_guide: CreativeQuestionGuide
    next_curiosity_topics: List[str] = Field(description="이어서 탐구할 다음 호기심 주제·질문 정확히 3개")


class ActivityFeasibility(BaseModel):
    branch: str = Field(
        description="safe_home_experiment | observation_instead | dialogue_only 중 하나"
    )
    reason: str = Field(description="한 줄 판단 근거")


class ActivityGuide(BaseModel):
    title: str = Field(description="활동명")
    materials: List[str] = Field(description="준비물 (집에서 가능한 것)")
    steps: List[str] = Field(description="방법 단계 3~6개")
    safety_note: str = Field(description="반드시 보호자 동반 등 안전 문구")
    kind_label: str = Field(description="실험 | 관찰 놀이 | 대화·역할 놀이 등")


class ImageCardPrompt(BaseModel):
    prompt_en: str = Field(
        description="gpt-image-1.5용 영문 한 문단. 어린이 교육용 카드 일러스트. 글자·숫자 없음."
    )


class QuestionList(BaseModel):
    question_list: List[str] = Field(description="짧은 질문 문자열 목록")


class ImageNeedDecision(BaseModel):
    need_image: bool = Field(
        description="교육용 일러스트 카드가 대화·이해에 도움이 되면 true. 감정·민감 주제 등 그림이 불필요·부적절하면 false."
    )
    reason: str = Field(description="판단 한 줄 (건너뛸 때는 왜 불필요한지)")


class State(TypedDict, total=False):
    learner_id: str
    target_age: int
    child_interests: List[str]
    explanation_style: str
    question: str
    question_analysis: QuestionAnalysis
    study_subject: StudySubject
    prior_learning_notes: str
    coach_pack: ParentCoachPack
    activity_feasibility: ActivityFeasibility
    activity_guide: ActivityGuide
    image_card_url: str
    image_card_prompt_en: str
    image_card_prompts: List[str]
    image_card_paths: List[str]
    image_need: ImageNeedDecision
    child_reaction_note: str
    related_question_suggestions: List[str]


graph_builder = StateGraph(State)


In [43]:
def ensure_child_profile(state: State):
    """아이 연령(필수), 관심사·설명 스타일 기본값."""
    target_age = state.get("target_age")
    if target_age is None:
        target_age = interrupt({"prompt": "대상 아이 나이(숫자). 예: 4"})
    try:
        target_age_int = int(str(target_age).strip())
    except Exception:
        target_age_int = 4
    target_age_int = max(3, min(8, target_age_int))

    interests = state.get("child_interests")
    if interests is None:
        interests = []
    if isinstance(interests, str):
        interests = [x.strip() for x in interests.replace(",", " ").split() if x.strip()]

    style = (state.get("explanation_style") or "짧고 재밌게").strip() or "짧고 재밌게"

    return {
        "target_age": target_age_int,
        "child_interests": interests,
        "explanation_style": style,
    }


def analyze_question(state: State):
    """질문 유형·의도·부모에게 필요한 도움."""
    print(f"analyze_question => {state['question']}")
    ta = state.get("target_age", 4)
    interests = state.get("child_interests") or []
    style = state.get("explanation_style") or ""
    response = llm.with_structured_output(QuestionAnalysis).invoke(f"""
    당신은 부모를 돕는 교육 코치입니다. 아래는 부모가 적어 준 아이의 질문입니다.

    아이 질문:
    {state["question"]}

    아이 나이: {ta}세 (만 나이 기준으로 해석)
    아이가 좋아하는 것(있으면 비유·예시에 반영): {interests}
    부모가 원하는 설명 스타일: {style}

    규칙:
    - 과학 질문으로만 몰아가지 마세요. 감정·사회·민감 주제도 구분합니다.
    - question_type은 반드시 한 줄로: 과학·자연 / 감정·관계 / 사회·가치 / 민감·신체·죽음 등 / 일상·기타 중 가까운 것.
    - 민감 주제는 부모_need에 '조심스러운 대화, 정확한 정보 탐색, 아이 속도 존중' 등을 적습니다.
    """)
    return {"question_analysis": response}


def convert_study_subject(state: State):
    print(f"convert_study_subject => {state['question_analysis']}")
    qa = state["question_analysis"]
    response = llm.with_structured_output(StudySubject).invoke(f"""
    아이의 질문을 배우기 좋은 학습 주제로 정리합니다. 원 질문과 분석과 동떨어지면 안 됩니다.

    원 질문:
    {state["question"]}

    분석:
    - 유형: {qa.question_type}
    - 아이 의도: {qa.child_intent}
    - 부모에게 필요한 도움: {qa.parent_need}
    - 난이도 감각: {qa.difficulty_level}
    """)
    return {"study_subject": response}


def normalize_topic_key(title: str) -> str:
    return " ".join((title or "").lower().split())[:200]


def load_prior_learning(state: State) -> dict:
    study = state["study_subject"]
    topic_key = normalize_topic_key(study.title)
    learner_id = (state.get("learner_id") or "").strip()

    cur = conn.cursor()
    if learner_id:
        cur.execute(
            """
            SELECT study_title, core_concept, description_excerpt, created_at
            FROM learning_records
            WHERE topic_key = ? AND learner_id = ?
            ORDER BY id DESC LIMIT 5
            """,
            (topic_key, learner_id),
        )
    else:
        cur.execute(
            """
            SELECT study_title, core_concept, description_excerpt, created_at
            FROM learning_records
            WHERE topic_key = ?
            ORDER BY id DESC LIMIT 5
            """,
            (topic_key,),
        )
    rows = cur.fetchall()
    if not rows:
        return {"prior_learning_notes": ""}

    lines = []
    for study_title, core_concept, excerpt, created_at in rows:
        excerpt_short = ((excerpt or "").replace("\n", " "))[:120]
        lines.append(
            f"- ({created_at}) {study_title} — {(core_concept or '')[:40]}… 요약: {excerpt_short}"
        )
    notes = (
        "【이전 호기심 기록】비슷한 주제를 예전에 다뤘을 수 있습니다.\n"
        + "\n".join(lines)
    )
    return {"prior_learning_notes": notes}


def generate_parent_coach(state: State):
    print(f"generate_parent_coach => {state['study_subject'].title}")
    study = state["study_subject"]
    qa = state["question_analysis"]
    ta = state.get("target_age", 4)
    prior = (state.get("prior_learning_notes") or "").strip()
    interests = state.get("child_interests") or []
    style = state.get("explanation_style") or "짧고 재밌게"

    prior_block = ""
    if prior:
        prior_block = f"\n{prior}\n\n예전에 비슷하게 이야기했다면 한두 문장으로 연결한 뒤 오늘 내용으로 이어가세요.\n"

    if ta <= 5:
        age_rules = """
    【만 5세 이하】전문어 금지 수준. 비유는 일상·좋아하는 것 위주. 문장 짧게."""
    elif ta <= 7:
        age_rules = """
    【만 6~7세】용어는 최소화, 나오면 한 줄 풀어서."""
    else:
        age_rules = """
    【만 8세 전후】조금 더 설명 가능하나 여전히 부모가 읽어 주기 쉬운 말투."""

    response = llm.with_structured_output(ParentCoachPack).invoke(f"""
    당신은 만 3~6세 부모를 위한 **호기심 학습 코치**입니다. 아이는 직접 타이핑하지 않습니다.

    아이 질문:
    {state["question"]}

    질문 분석:
    - 유형: {qa.question_type}
    - 아이 의도: {qa.child_intent}
    - 부모에게 필요한 것: {qa.parent_need}

    학습 주제: {study.title}
    핵심: {study.core_concept}
    목표: {study.learning_goal}
    키워드: {study.keywords}

    아이 나이: {ta}세 | 좋아하는 것: {interests} | 설명 스타일: {style}
    {prior_block}
    {age_rules}

    출력 규칙:
    - short_answer_30s: 지금 바로 말할 초간단 설명 (2~4문장).
    - story_answer_3min: 조금 더 긴 이야기 (여러 짧은 문단).
    - play_and_picture_tip: 그림·장난감·몸으로 설명하는 방법 한 덩어리.
    - parent_read_aloud_script: 부모가 **연속해서** 읽을 수 있는 대화형 스크립트. 아이 말투 vs 부모 말투가 섞여도 됨.
    - phrases_to_avoid / say_instead: '그냥 그런 거야' 류 **피할 말**과 **대신 말** 구체적으로.
    - follow_up_questions: 정확히 3개, 되물음 (대화가 끊기지 않게).
    - creative_question_guide: 관찰/상상/비교/탐구 각각 한 문장씩.
    - next_curiosity_topics: 이어서 탐구할 질문·주제 **정확히 3개** (다음 호기심).

    민감·감정 질문이면 과학 실험 대신 공감·안전·경계 존중을 우선하고, 확실치 않은 정보는 단정하지 마세요.
    """)

    return {"coach_pack": response}


def decide_activity_feasibility(state: State):
    study = state["study_subject"]
    qa = state["question_analysis"]
    print(f"decide_activity_feasibility => {qa.question_type}")

    response = llm.with_structured_output(ActivityFeasibility).invoke(f"""
    집에서 할 수 있는 활동 유형을 판단합니다.

    질문 유형: {qa.question_type}
    학습 주제: {study.title}
    핵심 개념: {study.core_concept}

    branch는 반드시 다음 중 하나만 (영어 snake_case 그대로):
    - safe_home_experiment: 집에서 재료로 안전하게 할 수 있는 간단 관찰·실험
    - observation_instead: 실험 대신 산책·창밖 관찰·그림 등으로 충분
    - dialogue_only: 실험보다 대화·역할극·책·공감이 우선 (감정·민감·사회 주제 등)

    한 줄 reason도 적으세요.
    """)
    return {"activity_feasibility": response}


def generate_activity_guide(state: State):
    study = state["study_subject"]
    coach = state["coach_pack"]
    fe = state["activity_feasibility"]
    ta = state.get("target_age", 4)
    print(f"generate_activity_guide => branch={fe.branch}")

    response = llm.with_structured_output(ActivityGuide).invoke(f"""
    부모가 아이와 집에서 할 **한 가지** 활동을 설계합니다.

    분기: {fe.branch}
    이유: {fe.reason}

    주제: {study.title}
    핵심: {study.core_concept}

    코치 요약(참고):
    - 30초 답: {coach.short_answer_30s[:400]}

    규칙:
    - safe_home_experiment: 준비물은 집에서 구하기 쉬운 것. 단계는 3~6개.
    - observation_instead: 산책·하늘 보기·그림 그리기 등 관찰·놀이 중심.
    - dialogue_only: 역할극 대사 예시, 함께 읽을 질문, 안전한 상상 놀이. 실험 강요 금지.

    safety_note에 반드시 보호자 동반·위험 행동 금지를 넣으세요.
    아이 나이 {ta}세에 맞추세요.
    """)
    return {"activity_guide": response}


def decide_image_need(state: State):
    """질문 성격에 따라 학습용 일러스트가 필요한지 판단 — 필요할 때만 이후 노드에서 Tool 호출."""
    qa = state["question_analysis"]
    study = state["study_subject"]
    coach = state["coach_pack"]
    fe = state["activity_feasibility"]
    print(f"decide_image_need => {qa.question_type}")

    response = llm.with_structured_output(ImageNeedDecision).invoke(f"""
    아이 질문과 맥락을 보고, **교육용 그림 카드(일러스트)를 이번에 생성할지** 판단합니다.

    질문 유형: {qa.question_type}
    학습 주제: {study.title}
    핵심: {study.core_concept}
    활동 분기: {fe.branch} — {fe.reason}

    30초 답 일부: {coach.short_answer_30s[:300]}

    need_image=true:
    자연·과학처럼 눈으로 비유하면 이해가 쉬운 주제, 또는 한 장면 일러스트가 호기심을 붙이는 경우.

    need_image=false 예:
    - 죽음, 폭력, 학대, 가난 등 민감 주제에서 그림이 불안·오해를 키울 수 있을 때
    - 관계·감정 중심이라 말·안식·포옹이 우선일 때
    - 한 장의 귀여운 삽화가 학습 효과나 안전 측면에서 불필요하다고 판단될 때

    reason은 한 줄로 명확히 적습니다.
    """)
    return {"image_need": response}


def skip_image_card(state: State):
    """이미지 Tool을 호출하지 않고 빈 결과만 채웁니다."""
    img = state.get("image_need")
    r = img.reason if img else ""
    print(f"skip_image_card => {r}")
    return {
        "image_card_prompt_en": "",
        "image_card_prompts": [],
        "image_card_paths": [],
        "image_card_url": "",
    }


def _image_bytes_extension(image_bytes: bytes) -> str:
    if image_bytes.startswith(b"\x89PNG\r\n\x1a\n"):
        return ".png"
    if image_bytes.startswith(b"\xff\xd8\xff"):
        return ".jpg"
    return ".png"


@tool
def generate_education_image_card(prompt_en: str) -> dict:
    """gpt-image-1.5로 이미지를 받아 디스크에 저장합니다."""
    prompt_en = (prompt_en or "").strip()[:4000]
    empty = {"image_card_prompts": [], "image_card_paths": []}
    if not prompt_en:
        return empty

    concept_id = hashlib.sha256(prompt_en.encode("utf-8")).hexdigest()[:16]

    try:
        client = OpenAI()
        img = client.images.generate(
            model="gpt-image-1.5",
            prompt=prompt_en,
            size="1024x1024",
            quality="low",
            n=1,
        )
        first = img.data[0]
        b64 = getattr(first, "b64_json", None)
        http_url = getattr(first, "url", None)

        if b64:
            image_bytes = base64.b64decode(b64)
            ext = _image_bytes_extension(image_bytes)
            filename = f"image_card_{concept_id}{ext}"
            with open(filename, "wb") as file:
                file.write(image_bytes)
            return {
                "image_card_prompts": [prompt_en],
                "image_card_paths": [filename],
            }

        if http_url:
            filename = f"image_card_{concept_id}.jpg"
            urllib.request.urlretrieve(http_url, filename)
            return {
                "image_card_prompts": [prompt_en],
                "image_card_paths": [filename],
            }

        return {"image_card_prompts": [prompt_en], "image_card_paths": []}
    except Exception as e:
        print(f"generate_education_image_card tool error: {e}")
        return {"image_card_prompts": [prompt_en], "image_card_paths": []}


learning_tools = [generate_education_image_card]


def create_image_card(state: State):
    study = state["study_subject"]
    coach = state["coach_pack"]
    ta = state.get("target_age", 4)

    print(f"create_image_card => {study.title}")

    spec = llm.with_structured_output(ImageCardPrompt).invoke(f"""
You write ONE English image-generation prompt for OpenAI gpt-image-1.5 (children's educational flashcard illustration).

Topic title: {study.title}
Core concept: {study.core_concept}

Korean coach content for a {ta}-year-old (ideas only, distill to visual scene):
{coach.parent_read_aloud_script[:1200]}

Rules:
- Single cute friendly scene, bright pastel, simple shapes, no text/letters/numbers in the image.
- No violence, horror, or scary realism.
- Square-friendly learning card composition.

Return structured output only.
""")
    prompt_en = spec.prompt_en.strip()

    out = generate_education_image_card.invoke({"prompt_en": prompt_en})
    if not isinstance(out, dict):
        out = {}

    prompts = out.get("image_card_prompts") or ([prompt_en] if prompt_en else [])
    paths = out.get("image_card_paths") or []
    local_path = paths[0] if paths else ""

    return {
        "image_card_prompt_en": prompt_en,
        "image_card_prompts": prompts,
        "image_card_paths": paths,
        "image_card_url": local_path,
    }


def save_learning_record(state: State) -> dict:
    """호기심 기록장 + 다음 호기심 추천 목록."""
    study = state["study_subject"]
    topic_key = normalize_topic_key(study.title)
    learner_id = (state.get("learner_id") or "").strip()
    question = state.get("question") or ""
    target_age = state.get("target_age")
    coach = state.get("coach_pack")
    excerpt = (coach.short_answer_30s[:800] if coach else "") or ""
    fe = state.get("activity_feasibility")
    ag = state.get("activity_guide")
    note = (state.get("child_reaction_note") or "").strip()

    img_need = state.get("image_need")
    outcome_summary = "코치패키지·활동"
    if img_need and getattr(img_need, "need_image", False):
        outcome_summary += "·이미지 생성"
    elif img_need:
        outcome_summary += f" | 이미지 생략:{getattr(img_need, 'reason', '')[:60]}"
    else:
        outcome_summary += "·이미지 미판단"
    if fe:
        outcome_summary += f" | 활동분기:{fe.branch}"
    if ag:
        outcome_summary += f" | 활동:{ag.title[:40]}"
    if note:
        outcome_summary += f" | 부모메모:{note[:80]}"

    qa = state.get("question_analysis")
    question_category = (qa.question_type or "").strip()[:200] if qa else ""

    conn.execute(
        """
        INSERT INTO learning_records
        (topic_key, learner_id, question, study_title, core_concept, target_age, description_excerpt, outcome_summary, question_category)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        """,
        (
            topic_key,
            learner_id,
            question,
            study.title,
            study.core_concept,
            target_age,
            excerpt,
            outcome_summary,
            question_category,
        ),
    )
    conn.commit()

    suggestions: List[str] = []
    if coach and getattr(coach, "next_curiosity_topics", None):
        suggestions = [str(x).strip() for x in coach.next_curiosity_topics if str(x).strip()]
    if len(suggestions) < 3:
        try:
            sug = llm.with_structured_output(QuestionList).invoke(f"""
            같은 맥락에서 아이가 이어서 물을 만한 질문을 3개만.
            주제: {study.title}
            핵심: {study.core_concept}
            """)
            for q in sug.question_list:
                q = str(q).strip()
                if q and q not in suggestions:
                    suggestions.append(q)
                if len(suggestions) >= 5:
                    break
        except Exception:
            pass

    return {"related_question_suggestions": suggestions[:5]}


In [44]:
graph_builder.add_node('ensure_child_profile', ensure_child_profile)
graph_builder.add_node('analyze_question', analyze_question)
graph_builder.add_node('convert_study_subject', convert_study_subject)
graph_builder.add_node('load_prior_learning', load_prior_learning)
graph_builder.add_node('generate_parent_coach', generate_parent_coach)
graph_builder.add_node('decide_activity_feasibility', decide_activity_feasibility)
graph_builder.add_node('generate_activity_guide', generate_activity_guide)
graph_builder.add_node('decide_image_need', decide_image_need)
graph_builder.add_node('create_image_card', create_image_card)
graph_builder.add_node('skip_image_card', skip_image_card)
graph_builder.add_node('save_learning_record', save_learning_record)


def route_image_need(state: State) -> str:
    img = state.get("image_need")
    if img is not None and getattr(img, "need_image", False):
        return "create_image_card"
    return "skip_image_card"


graph_builder.add_edge(START, 'ensure_child_profile')
graph_builder.add_edge('ensure_child_profile', 'analyze_question')
graph_builder.add_edge('analyze_question', 'convert_study_subject')
graph_builder.add_edge('convert_study_subject', 'load_prior_learning')
graph_builder.add_edge('load_prior_learning', 'generate_parent_coach')
graph_builder.add_edge('generate_parent_coach', 'decide_activity_feasibility')
graph_builder.add_edge('decide_activity_feasibility', 'generate_activity_guide')
graph_builder.add_edge('generate_activity_guide', 'decide_image_need')
graph_builder.add_conditional_edges(
    "decide_image_need",
    route_image_need,
    {
        "create_image_card": "create_image_card",
        "skip_image_card": "skip_image_card",
    },
)
graph_builder.add_edge('create_image_card', 'save_learning_record')
graph_builder.add_edge('skip_image_card', 'save_learning_record')
graph_builder.add_edge('save_learning_record', END)

memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)

In [45]:
config = {"configurable": {"thread_id": "demo"}}

# 한 번의 invoke: 질문 분석 → 코치 패키지 → 활동 분기 → 이미지 → SQLite 저장
result = graph.invoke({
    "question": "비는 왜 내리는거야 ?",
    "target_age": 8,
    "child_interests": ["우주", "자동차"],
    "explanation_style": "짧고 재밌게",
    # "child_reaction_note": "아이가 창밖을 가리키며 물어봄",  # 선택: 기록에만 반영
    # "learner_id": "kid-01",
}, config=config)
result

analyze_question => 비는 왜 내리는거야 ?
convert_study_subject => question_type='과학·자연' child_intent='비가 내리는 이유에 대해 알고 싶어하는 아이의 호기심 반영' parent_need='짧고 재밌게 설명해주길 원함' difficulty_level='보통'
generate_parent_coach => 비 내림의 원리 이해하기
decide_activity_feasibility => 과학·자연
generate_activity_guide => branch=safe_home_experiment
decide_image_need => 과학·자연
create_image_card => 비 내림의 원리 이해하기


{'target_age': 8,
 'child_interests': ['우주', '자동차'],
 'explanation_style': '짧고 재밌게',
 'question': '비는 왜 내리는거야 ?',
 'question_analysis': QuestionAnalysis(question_type='과학·자연', child_intent='비가 내리는 이유에 대해 알고 싶어하는 아이의 호기심 반영', parent_need='짧고 재밌게 설명해주길 원함', difficulty_level='보통'),
 'study_subject': StudySubject(title='비 내림의 원리 이해하기', core_concept='비는 구름 속의 수증기가 모여서 물방울이 되고, 이 물방울이 한 곳에 모여 무겁게 되면 떨어진다.', learning_goal='비가 왜 내리는지에 대한 과학적 원리를 이해하고, 구름과 물방울의 관계를 통해 자연현상의 기본적인 흐름을 배운다.', keywords=['과학', '자연', '비', '구름', '수증기', '물방울']),
 'prior_learning_notes': '',
 'coach_pack': ParentCoachPack(short_answer_30s='비는 구름에 있는 수증기들이 모여서 물방울이 돼서 떨어지는 거야. 그렇게 물방울이 무겁게 되면, 마치 공이 떨어지는 것처럼 비가 내리는 거야!', story_answer_3min='한 번, 하늘에 있는 구름들은 마치 큰 솥처럼 수증기라는 물을 담고 있었어. 이 수증기들이 서로 손잡고 점점 커지다가, 물방울이 되어 아침에 수분이 많은 날이 오면 한 곳에 모여서 무겁게 되지! 무거워진 물방울들은 결국 하늘에서 떨어져서 우리에게 비가 되어 내리게 돼. 그래서 길고 긴 하늘의 여행을 마치고 비가 되어 땅에 가는 거야. 그 덕분에 나무와 꽃들도 물을 마시고 더욱 푸르게 자라지!', play_and_picture_tip='아이에게 큰 그릇에 물을 담고, 그것이 한가득 차오를 때 물이 넘치는 모습

In [46]:
# 주요 결과 키:
# result["coach_pack"], result["activity_guide"], result["image_need"] (need_image / reason)
# 이미지는 질문 성격상 불필요하면 생략되며 paths가 비고 skip 이유가 reason에 있습니다.
# result["related_question_suggestions"]

# --- 호기심 기록: 카테고리(질문 유형)별 조회 (위에서 conn 이미 생성된 뒤 실행) ---
# list_question_categories()                          # 전체 아이 기준
# list_question_categories("kid-01")                # learner_id 지정
# fetch_learning_records(category="과학·자연")       # 카테고리 필터
# fetch_learning_records("kid-01", "감정·관계", 50)